In [3]:

from langgraph.graph.message import add_messages
from langgraph.graph import START, StateGraph, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

In [7]:
load_dotenv()

True

In [8]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [9]:
model = ChatOpenAI(
    model='gpt-4o-mini'
)

In [10]:
def chat_node(state: ChatState):
    messages = state['messages']

    response = model.invoke(messages)

    return {
        'messages': [response]
    }


In [5]:
graph = StateGraph(ChatState)

In [11]:
graph.add_node('chat_node', chat_node)

In [12]:
graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

In [13]:
chatbot = graph.compile()

In [14]:
initial_state = {
    'messages': [HumanMessage(content="What is the capital of India?")]
}

In [15]:
chatbot.invoke(initial_state)

{'messages': [HumanMessage(content='What is the capital of India?', additional_kwargs={}, response_metadata={}, id='26613553-e6d4-4340-b975-989985ae3500'),
  AIMessage(content='The capital of India is New Delhi.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 14, 'total_tokens': 22, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_629ce8fe0f', 'id': 'chatcmpl-EnttjYufNJ5VBDHVzIcs8wOayJyZ1', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None, 'normalized_finish_reason': 'stop'}, id='lc_run--01a20fd0-780e-7724-bab6-96fb15b07e8c-0', tool_calls=[], invalid_tool_calls=[]

In [ ]:
while True:
    user_input = input("Type Here:\n")
    print(f"YOU: {user_input}")
    if user_input.strip().lower() in {'exit', 'quit', 'bye'}:
        break

    response = chatbot.invoke({
        'messages': [HumanMessage(content=user_input)]
    })

    print(f"AI RESPONSE: {response['messages'][-1].content}")


